# Notebook 1 – Data Ingestion & Feature Engineering

This notebook combines:
- **Data Ingestion** from ADLS/CSV into Spark tables
- **Data Cleaning & Feature Engineering**, including VADER sentiment and lag/lead targets

It produces `project_clean.final_features` and `project_model.train_data` / `project_model.test_data` for modelling.


In [0]:
# Configuration: USING THE CORRECT 'landing' CONTAINER 
CONTAINER_PATH = "abfss://landing@projectbigdata.dfs.core.windows.net"
RAW_TWEET_FILE_PATH = f"{CONTAINER_PATH}/stocktweet/stocktweet.csv"
TARGET_TWEET_TABLE = "project_raw.raw_tweets" # Saving in the project_raw schema

# 1. Create a Schema (Database) for raw data in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS project_raw;")

# 2. Read the raw CSV data 
print(f"Reading raw tweet data from: {RAW_TWEET_FILE_PATH}")
df_tweets = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("multiLine", "true") \
    .load(RAW_TWEET_FILE_PATH)

# 3. Write the data to a Delta Lake table
df_tweets.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TWEET_TABLE)

print(f"Successfully created Delta table: {TARGET_TWEET_TABLE}")
spark.sql(f"SELECT COUNT(*) AS total_tweets, min(date) as min_date, max(date) as max_date FROM {TARGET_TWEET_TABLE}").show()
import re

def clean_column_name(col):
    """Replaces spaces with underscores and removes invalid characters."""
    # Replace spaces with underscore
    cleaned_col = col.replace(" ", "_")
    # Remove any other non-alphanumeric characters (like the period in 'Adj. Close' if present)
    cleaned_col = re.sub(r'[^\w]', '', cleaned_col)
    return cleaned_col
# --- Configuration: USING THE CORRECT 'landing' CONTAINER ---
CONTAINER_PATH = "abfss://landing@projectbigdata.dfs.core.windows.net"
RAW_PRICE_FOLDER_PATH = f"{CONTAINER_PATH}/stockprice/" 
TARGET_PRICE_TABLE = "project_raw.raw_prices"

# 1. Read all price CSV files from the folder
print(f"Reading all price CSVs from folder: {RAW_PRICE_FOLDER_PATH}")
df_prices = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(RAW_PRICE_FOLDER_PATH)


# Get the list of current column names
existing_cols = df_prices.columns
# Create a dictionary of old_name: new_name mappings
col_map = {col: clean_column_name(col) for col in existing_cols}

# Apply the renames to the DataFrame
for old_name, new_name in col_map.items():
    df_prices = df_prices.withColumnRenamed(old_name, new_name)

# 2. Write the data to a Delta Lake table
df_prices.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_PRICE_TABLE)

print(f"Successfully created Delta table: {TARGET_PRICE_TABLE}")
spark.sql(f"SELECT * FROM {TARGET_PRICE_TABLE} LIMIT 5").show()
from pyspark.sql.functions import col, split, element_at, regexp_replace

#  Configuration 
CONTAINER_PATH = "abfss://landing@projectbigdata.dfs.core.windows.net"
RAW_PRICE_FOLDER_PATH = f"{CONTAINER_PATH}/stockprice/" 
TARGET_PRICE_TABLE = "project_raw.raw_prices"

# 1. Read all price CSV files
print(f"Reading all price CSVs from: {RAW_PRICE_FOLDER_PATH}")
df_prices = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(RAW_PRICE_FOLDER_PATH)


# We explicitly select all existing columns ("*") AND the file_path from the hidden _metadata column
df_prices = df_prices.select("*", col("_metadata.file_path").alias("file_path"))

# 2. Extract Ticker from the file_path
df_prices = df_prices.withColumn("filename", element_at(split("file_path", "/"), -1)) \
                     .withColumn("Ticker", regexp_replace("filename", r"\.csv$", ""))

# 3. Clean other Column Names (Replace spaces with underscores)
for col_name in df_prices.columns:
     
    clean_name = col_name.replace(" ", "_").replace(".", "")
    df_prices = df_prices.withColumnRenamed(col_name, clean_name)

# 4. Write the corrected data to Delta

df_prices_final = df_prices.drop("file_path", "filename")

df_prices_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_PRICE_TABLE)

print(f"Successfully corrected and saved Delta table: {TARGET_PRICE_TABLE}")
spark.sql(f"SELECT Ticker, Date, Close FROM {TARGET_PRICE_TABLE} LIMIT 5").show()

Reading raw tweet data from: abfss://landing@projectbigdata.dfs.core.windows.net/stocktweet/stocktweet.csv
Successfully created Delta table: project_raw.raw_tweets
+------------+----------+----------+
|total_tweets|  min_date|  max_date|
+------------+----------+----------+
|       10000|01/01/2020|31/12/2020|
+------------+----------+----------+

Reading all price CSVs from folder: abfss://landing@projectbigdata.dfs.core.windows.net/stockprice/
Successfully created Delta table: project_raw.raw_prices
+----------+-----------------+-----------------+-----------------+-----------------+-----------------+----------+------+
|      Date|             Open|             High|              Low|            Close|        Adj_Close|    Volume|Ticker|
+----------+-----------------+-----------------+-----------------+-----------------+-----------------+----------+------+
|2019-12-31|3215.179931640625|3231.719970703125|3212.030029296875|3230.780029296875|3230.780029296875|2894760000|  NULL|
|2020-01-

---

In [0]:
%pip install vaderSentiment
dbutils.library.restartPython()

import re
from pyspark.sql.functions import col, to_date, avg, udf
from pyspark.sql.types import DoubleType, DateType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Initialize the VADER analyzer and create the UDF
analyzer = SentimentIntensityAnalyzer()

def get_vader_compound_score(text):
    """Calculates the VADER compound score for a given text."""
    if text is None or text.strip() == "":
        return 0.0
    # VADER is especially good for social media and returns a compound score between -1 (extreme negative) and +1 (extreme positive)
    return analyzer.polarity_scores(text)['compound']

# Register the Python function as a Spark UDF

get_vader_compound_udf = udf(get_vader_compound_score, DoubleType())

# 2. Load Raw Tweets
print("Loading raw tweets and calculating sentiment...")
df_tweets = spark.table("project_raw.raw_tweets")

# 3. Clean Date and Calculate Sentiment
df_sentiments = df_tweets.select(
    # Normalize the date string 'MM/dd/yyyy' (e.g., '01/01/2020') to a proper DateType
    to_date(col("date"), "MM/dd/yyyy").alias("trade_date"),
    col("ticker"),
    # Calculate the compound sentiment score for each tweet
    get_vader_compound_udf(col("tweet")).alias("compound_score")
).filter(col("trade_date").isNotNull()) 

# 4. Aggregate to Daily Average Sentiment

df_daily_sentiment = df_sentiments.groupBy("trade_date", "ticker").agg(
    avg("compound_score").alias("daily_avg_sentiment")
).orderBy("trade_date", "ticker")

# 5. Create clean schema and save the first feature table
spark.sql("CREATE SCHEMA IF NOT EXISTS project_clean;")

TARGET_DAILY_SENTIMENT_TABLE = "project_clean.daily_sentiment"
df_daily_sentiment.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_DAILY_SENTIMENT_TABLE)

print(f"Successfully created Daily Sentiment Feature table: {TARGET_DAILY_SENTIMENT_TABLE}")
spark.sql(f"SELECT * FROM {TARGET_DAILY_SENTIMENT_TABLE} WHERE ticker = 'AMZN' ORDER BY trade_date LIMIT 5").show()
from pyspark.sql.functions import to_date, col

TARGET_PRICE_TABLE = "project_raw.raw_prices"
TARGET_FINAL_TABLE = "project_clean.final_features"

# 1. Load and Prepare Price Data

df_prices = spark.table(TARGET_PRICE_TABLE).select(
  
    to_date(col("Date")).alias("trade_date"), 
    col("Ticker").alias("ticker"),
    col("Close").alias("closing_price")
).filter(col("trade_date").isNotNull())


# 2. Load Sentiment Data
df_daily_sentiment = spark.table(TARGET_DAILY_SENTIMENT_TABLE)


# 3. Join Sentiment (Exogenous Variable) with Price Data
df_final_features = df_prices.join(
    df_daily_sentiment,
    on=["trade_date", "ticker"],
    how="left" # Use left join to keep all price data, even if a day had no tweets
).fillna(0.0, subset=['daily_avg_sentiment']) # Fill NULL sentiment (days with no tweets) with 0.0 (neutral)


# 4. Save Final Feature Table
print("Joining data and saving final feature table...")
df_final_features.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_FINAL_TABLE)

print(f"Successfully created Final Feature table: {TARGET_FINAL_TABLE}")
spark.sql(f"SELECT * FROM {TARGET_FINAL_TABLE} WHERE ticker = 'TSLA' ORDER BY trade_date DESC LIMIT 5").show()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/126.0 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Loading raw tweets and calculating sentiment...
Successfully created Daily Sentiment Feature table: project_clean.daily_sentiment
+----------+------+-------------------+
|trade_date|ticker|daily_avg_sentiment|
+----------+------+-------------------+
|2020-01-01|  AMZN|             0.3818|
|2020-01-02|  AMZN|             0.5499|
|2020-01-05|  AMZN|0.21951666666666667|
|2020-01-07|  AMZN| 0.5923750000000001|
|2020-01-08|  AMZN|             0.4939|
+----------+------+-------------------+

Joining data and saving final feature table...
Successfully created Final Feature table: project_clean.final_features
+----------+------+------------------+-------------------+---------------+---------------+---------------+-----------

In [0]:
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window
import pyspark.sql.functions as F

TARGET_FINAL_FEATURES_TABLE = "project_clean.final_features"

# Load the joined data
df_final = spark.table(TARGET_FINAL_FEATURES_TABLE)

# 1. Define the Window Specification

window_spec = Window.partitionBy("ticker").orderBy("trade_date")

# 2. Create Lagged Features

df_features = df_final \
    .withColumn("lag_1_day_close", lag(col("closing_price"), 1).over(window_spec)) \
    .withColumn("lag_3_day_close", lag(col("closing_price"), 3).over(window_spec)) \
    .withColumn("lag_7_day_close", lag(col("closing_price"), 7).over(window_spec))

# 3. Create Rolling Averages 
df_features = df_features \
    .withColumn("ma_5_day_close", F.avg(col("closing_price")).over(window_spec.rowsBetween(-4, 0))) \
    .withColumn("ma_20_day_close", F.avg(col("closing_price")).over(window_spec.rowsBetween(-19, 0)))

# 4. Fill NULLs created by the Lag function 
df_features = df_features.fillna(0.0, subset=["lag_1_day_close", "lag_3_day_close", "lag_7_day_close"])

# Overwrite the features table with the new lagged columns
df_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_FINAL_FEATURES_TABLE)

print(f"Step 3: Lagged features (lag_1/3/7_day_close) and Moving Averages added to: {TARGET_FINAL_FEATURES_TABLE}")
spark.sql(f"SELECT ticker, trade_date, closing_price, lag_1_day_close FROM {TARGET_FINAL_FEATURES_TABLE} ORDER BY ticker, trade_date DESC LIMIT 5").show()
from pyspark.sql.functions import lead, col
from pyspark.sql.window import Window

TARGET_FINAL_FEATURES_TABLE = "project_clean.final_features"

# Load the features data
df_features = spark.table(TARGET_FINAL_FEATURES_TABLE)

# 1. Define the Window Specification (same as before)
window_spec = Window.partitionBy("ticker").orderBy("trade_date")

# 2. Create Target Variables using LEAD

df_final_with_targets = df_features \
    .withColumn("target_1_day", lead(col("closing_price"), 1).over(window_spec)) \
    .withColumn("target_3_day", lead(col("closing_price"), 3).over(window_spec)) \
    .withColumn("target_7_day", lead(col("closing_price"), 7).over(window_spec))

# 3. Save the final dataset
df_final_with_targets.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_FINAL_FEATURES_TABLE)

print(f"Step 4: Target variables (target_1/3/7_day) added to the final dataset.")
spark.sql(f"SELECT ticker, trade_date, closing_price, target_1_day, target_7_day FROM {TARGET_FINAL_FEATURES_TABLE} ORDER BY ticker, trade_date DESC LIMIT 5").show()
from pyspark.sql.functions import col

TARGET_FINAL_FEATURES_TABLE = "project_clean.final_features"

# Load the data with targets
df_final = spark.table(TARGET_FINAL_FEATURES_TABLE)

# 1. Drop rows where any of the target columns are NULL

df_ready_to_train = df_final.na.drop(subset=["target_1_day", "target_3_day", "target_7_day"])

# 2. Drop the first few rows for each ticker where lagged features are NULL (optional, but cleaner)
df_ready_to_train = df_ready_to_train.filter(col("lag_7_day_close").isNotNull())


df_ready_to_train.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_FINAL_FEATURES_TABLE)

print(f"Successfully dropped NULL target rows. Data is ready for training.")
print(f"Total rows ready for training: {df_ready_to_train.count()}")

Step 3: Lagged features (lag_1/3/7_day_close) and Moving Averages added to: project_clean.final_features
+-------+----------+-----------------+-----------------+
| ticker|trade_date|    closing_price|  lag_1_day_close|
+-------+----------+-----------------+-----------------+
|%5EGSPC|2020-12-31|3756.070068359375|  3732.0400390625|
|%5EGSPC|2020-12-30|  3732.0400390625|  3727.0400390625|
|%5EGSPC|2020-12-29|  3727.0400390625|3735.360107421875|
|%5EGSPC|2020-12-28|3735.360107421875| 3703.06005859375|
|%5EGSPC|2020-12-24| 3703.06005859375|3690.010009765625|
+-------+----------+-----------------+-----------------+

Step 4: Target variables (target_1/3/7_day) added to the final dataset.
+-------+----------+-----------------+-----------------+------------+
| ticker|trade_date|    closing_price|     target_1_day|target_7_day|
+-------+----------+-----------------+-----------------+------------+
|%5EGSPC|2020-12-31|3756.070068359375|             NULL|        NULL|
|%5EGSPC|2020-12-30|  3732.04